In [1]:
import pandas as pd
from utils import min_max_normalize
import numpy as np

# Carregar dados e preparar variáveis

In [2]:
def accumulate(row, df):
    columns_to_cumsum = [
        "tempo_atuacao_dias"
    ]

    columns_to_avg = ["tempo_atuacao_percent"]

    columns_to_do_nothing = ["mand_dep_estadual", "mand_ver", "mand_sen"]

    id_parl = row["id"]
    id_legis = row["idLegislatura"]

    df_filtered = df[(df["id"] == id_parl) & (df["idLegislatura"] <= id_legis)]

    for col in columns_to_cumsum:
        row[f"{col}_cum"] = df_filtered[col].sum()

    for col in columns_to_avg:
        row[f"{col}_cum"] = df_filtered[col].mean()

    for col in columns_to_do_nothing:
        row[f"{col}_cum"] = row[col]

    return row


In [3]:
df_res = pd.read_excel(r"E:\repos\pessoal\redem-index\outputs\df_res.xlsx")

df_res = df_res.apply(
    accumulate, axis=1, args=(df_res,)
)

df_res['tempo_atuacao_dias_cum_ln'] = np.log(df_res['tempo_atuacao_dias_cum'] + 1)

df_res["mesa_cum"] = df_res["mesa_1_cum"] + df_res["mesa_2_cum"]


df_res["mandatos_cum"] = (
    df_res["mand_dep_estadual_cum"]
    + df_res["mand_sen_cum"]
    + df_res["mand_dep_estadual_cum"]
)



vars = [
    ("tempo_atuacao_dias_cum_ln", r"$TempoAtuacao$"),
    ("relatorias_ln_cum", r"$ln(Relatoria)$"),
    ("pos_lider_cum", r"$PosicaoLider$"),
    ("pos_comiss_pr_cum", r"$PresidenciaComissao$"),
    ("mesa_cum", r"$Mesa$"),
    ("mandatos_cum", r"$Mandatos$"),
    ("fid_gerais_cum", r"$FidelidadeGerais$"),
]

cols = [v[0] for v in vars]

df_res = df_res.fillna(0)

df_res[cols].describe()

df_final = df_res.iloc[:, 0:8].join(df_res[cols])

# Normalizando e aplicando os pesos

In [5]:
df_final

,id,idLegislatura,ultimoStatus.nome,ultimoStatus.siglaPartido,sexo,dataNascimento,escolaridade,idade_posse,tempo_atuacao_dias_cum_ln,relatorias_ln_cum,pos_lider_cum,pos_comiss_pr_cum,mesa_cum,mandatos_cum,fid_gerais_cum
0,1500,48,MÁRIO LIMA,PMDB,M,1935-02-19,Secundário,51,7.286876,0.693147,0,0,0,0,0
1,1501,48,MAURÍCIO FRUET,PMDB,M,1939-08-12,Superior,47,7.286876,0.000000,0,0,0,4,0
2,1502,48,OSWALDO ALMEIDA,PL*,M,1933-10-22,Superior,53,7.286876,0.000000,0,0,0,0,0
3,1503,48,OSWALDO LIMA FILHO,PMDB,M,1921-04-26,Superior,65,7.286876,0.000000,0,0,0,4,0
4,1504,48,TITO COSTA,PMDB,M,1922-12-31,Superior,64,7.286876,1.098612,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6245,229225,57,Missionária Michele Collins,PP,F,1977-10-17,0,45,4.454347,0.000000,0,0,0,0,0
6246,229259,57,Pedro Tourinho,PT,M,1981-12-19,0,41,4.394449,0.000000,0,0,0,0,0
6247,229333,57,Gláucia Santiago,PL,F,1960-02-20,0,62,4.382027,0.000000,0,0,0,0,0
6248,229432,57,Luiz Fernando Vampiro,MDB,M,1973-11-07,Superior,49,4.290459,0.000000,0,0,0,0,0


In [6]:
# normalizar
for col, label in vars:
    df_final[f"{col}_norm"] = min_max_normalize(df_final[col])

# appply weights
weights = {
    "tempo_atuacao_dias_cum_ln_norm": 1,
    "relatorias_ln_cum_norm": 2,
    "pos_lider_cum_norm": 4,
    "pos_comiss_pr_cum_norm": 3,
    "mesa_cum_norm": 5,
    "mandatos_cum_norm": 1,
    "fid_gerais_cum_norm": 1,
}
for var_name, weight in weights.items():
    df_final[f"{var_name}_w"] = df_final[var_name] * weight

df_final.iloc[:, 8:].describe().T.sort_index()

,count,mean,std,min,25%,50%,75%,max
fid_gerais_cum,6250.0,1.297280,1.340057,0.0,0.000000,1.000000,2.000000,7.000000
fid_gerais_cum_norm,6250.0,0.185326,0.191437,0.0,0.000000,0.142857,0.285714,1.000000
fid_gerais_cum_norm_w,6250.0,0.185326,0.191437,0.0,0.000000,0.142857,0.285714,1.000000
mandatos_cum,6250.0,1.248800,2.139633,0.0,0.000000,0.000000,2.000000,14.000000
mandatos_cum_norm,6250.0,0.089200,0.152831,0.0,0.000000,0.000000,0.142857,1.000000
mandatos_cum_norm_w,6250.0,0.089200,0.152831,0.0,0.000000,0.000000,0.142857,1.000000
mesa_cum,6250.0,0.026560,0.210004,0.0,0.000000,0.000000,0.000000,5.000000
mesa_cum_norm,6250.0,0.005312,0.042001,0.0,0.000000,0.000000,0.000000,1.000000
mesa_cum_norm_w,6250.0,0.026560,0.210004,0.0,0.000000,0.000000,0.000000,5.000000
pos_comiss_pr_cum,6250.0,0.254560,0.600314,0.0,0.000000,0.000000,0.000000,5.000000


# Calculando dimensões e índice final

In [7]:
dimensoes = {
    "dim_comprometimento": [
        "mesa_cum_norm_w",
        "pos_lider_cum_norm_w",
        "pos_comiss_pr_cum_norm_w",
        "relatorias_ln_cum_norm",
    ],
    "dim_carreira": [
        "mandatos_cum_norm_w",
        "fid_gerais_cum_norm_w",
        "tempo_atuacao_dias_cum_ln_norm_w",
    ],
}

df_final["dim_comprometimento"] = (
    df_final["mesa_cum_norm_w"]
    + df_final["pos_lider_cum_norm_w"]
    + df_final["pos_comiss_pr_cum_norm_w"]
    + df_final["relatorias_ln_cum_norm"]
)

df_final["dim_carreira"] = (
    df_final["mandatos_cum_norm_w"]
    + df_final["fid_gerais_cum_norm_w"]
    + df_final["tempo_atuacao_dias_cum_ln_norm_w"]
)

for col in ["dim_comprometimento", "dim_carreira"]:
    df_final[f"{col}_norm"] = min_max_normalize(df_final[col])

df_final["ipp"] = (df_final["dim_comprometimento_norm"] + df_final["dim_carreira_norm"]) / 2

In [8]:
df_final['tempo_atuacao_dias_cum'] = df_res['tempo_atuacao_dias_cum']

In [10]:
df_final.to_excel('./outputs/df_ipp_v2f.xlsx')